In [1]:
import pyvisa
from pyvisa.constants import Parity, StopBits

# Adjust to your RS232 settings
PORT =  "ASRL7::INSTR"   # <-- change to match your RS232 port
BAUD = 9600    # <-- change to your baud rate
VISA_LIBRARY = "C:/Windows/System32/visa64.dll"  # "auto" or r"C:/Windows/System32/visa64.dll", or "@py"
RS232_PORT_NAME = "ASRL7::INSTR"  # e.g. "ASRL5::INSTR"
BAUD_RATE = 9600  # e.g. 9600
rm = pyvisa.ResourceManager(VISA_LIBRARY)
inst = rm.open_resource(RS232_PORT_NAME)

# Configure serial settings to match the lock-in
inst.baud_rate = BAUD
inst.parity = Parity.even
inst.data_bits = 7
inst.stop_bits = StopBits.one
inst.timeout = 5000  # ms



In [2]:
import time
import pyvisa
from pyvisa import errors as visa_errors
from pyvisa.constants import Parity, StopBits, VI_ERROR_TMO


INTER_CHAR_DELAY = 0.02  # 20 ms; adjust 10–50 ms if needed

def open_inst(port, baud, parity=Parity.even, data_bits=7, stop_bits=StopBits.one):
    rm = pyvisa.ResourceManager()
    inst = rm.open_resource(port)
    inst.baud_rate = baud
    inst.parity = parity
    inst.data_bits = data_bits
    inst.stop_bits = stop_bits
    inst.write_termination = None
    inst.read_termination  = None
    inst.timeout = 5000  # ms per VISA read
    return inst

def read_until_prompt(inst, endbytes=(b'*', b'?'), overall_timeout_s=3.0, max_bytes=512):
    buf = bytearray()
    prompt = None
    old_to = inst.timeout
    inst.timeout = 100  # short per-read so we can loop
    deadline = time.time() + overall_timeout_s
    try:
        while time.time() < deadline and len(buf) < max_bytes:
            try:
                b = inst.read_bytes(1)
            except visa_errors.VisaIOError as e:
                if getattr(e, "error_code", None) == VI_ERROR_TMO:
                    continue
                raise
            if not b:
                continue
            if b in endbytes:
                prompt = b
                break
            if b not in (b'\r', b'\n'):
                buf.extend(b)
        return bytes(buf), prompt
    finally:
        inst.timeout = old_to

def print_result(label, payload, prompt):
    print(f"{label} -> payload bytes: {payload!r}  | decoded: {payload.decode('utf-8', 'replace')!r}  | prompt: {prompt!r}")


In [14]:
# ---------- Test Mode A: one-shot write ----------
inst = open_inst(PORT, BAUD, parity=Parity.even, data_bits=7, stop_bits=StopBits.one)
try:
    print("Mode A: one-shot 'ST\\r'")
    inst.write_raw(b"ST\r")
    payload, prompt = read_until_prompt(inst)
    print_result("Mode A", payload, prompt)
finally:
    inst.close()

Mode A: one-shot 'ST\r'
Mode A -> payload bytes: b'ST'  | decoded: 'ST'  | prompt: None


In [ ]:



# ---------- Test Mode B: char-by-char with small delay ----------
inst = open_inst(PORT, BAUD, parity=Parity.even, data_bits=7, stop_bits=StopBits.one)
try:
    print("\nMode B: char-by-char 'S','T','\\r' with small delay")
    inst.write_raw(b"S")
    time.sleep(INTER_CHAR_DELAY)
    inst.write_raw(b"T")
    time.sleep(INTER_CHAR_DELAY)
    inst.write_raw(b"\r")
    payload, prompt = read_until_prompt(inst)
    print_result("Mode B", payload, prompt)
finally:
    inst.close()



Mode B: char-by-char 'S','T','\r' with small delay
Mode B -> payload bytes: b'ST1'  | decoded: 'ST1'  | prompt: b'*'


In [10]:
_INTER_CHAR_DELAY = 0.02        # 20 ms worked in your Jupyter test
_OVERALL_RD_TIMEOUT_S = 3.0
     # overall time to wait for * or ?
def _inst_cmd(inst, cmd: str):
    """
    Send `cmd` with a small inter-character delay, then read bytes until the
    instrument prompt '*' (OK) or '?' (error). Returns (payload_text, status_byte).
    We strip the echoed command text (e.g., 'ST') so callers see just the data.
    """
    # --- Write: char-by-char without echo reads ---
    for ch in cmd:
        inst.write_raw(ch.encode('ascii'))
        time.sleep(_INTER_CHAR_DELAY)
    inst.write_raw(b'\r')

    # --- Read: 1 byte at a time until '*' or '?' (ignore CR/LF) ---
    # Use short per-read timeout to implement our own overall deadline.
    old_timeout = inst.timeout
    inst.timeout = 100  # ms per read attempt
    deadline = time.time() + _OVERALL_RD_TIMEOUT_S

    buf = bytearray()
    prompt = None
    try:
        while time.time() < deadline:
            try:
                b = inst.read_bytes(1)
            except visa_errors.VisaIOError as e:
                if getattr(e, "error_code", None) == VI_ERROR_TMO:
                    # no byte this slice; keep looping until deadline
                    continue
                raise
            if not b:
                continue
            if b in (b'*', b'?'):
                prompt = b
                break
            if b not in (b'\r', b'\n'):
                buf.extend(b)
    finally:
        inst.timeout = old_timeout

    # Decode and strip the echoed command (device echoes 'CMD' before data)
    text = buf.decode('utf-8', errors='ignore')
    if text.upper().startswith(cmd.upper()):
        text = text[len(cmd):].lstrip()

    # Mimic the old API's (response, status_byte) signature.
    # Callers in this file ignore status; return 1 for "done".
    return text, 1


In [14]:
channel = 1
sign = '+'
payload = "0"

cmd = f"DAC{channel}{sign}{payload}"

_inst_cmd(inst,cmd)

('', 1)